<a href="https://colab.research.google.com/github/Shyam-Singh-Bhargaw/DIABETIC-RETINOPATHY-CLASSIFICATION/blob/main/videos_to__screenshot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install youtube-transcript-api transformers torch reportlab pytube opencv-python

**Basic Version (first script, without batching or translation): Lacks TranscriptDataset, DataLoader, and Hindi-to-English translation.
Advanced Version (second script, with batching and translation): Includes TranscriptDataset, DataLoader, and robust transcript handling (e.g., for 5DSnWRHrv6U).**

In [15]:
import youtube_transcript_api
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
import re
import os
import torch
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet
from google.colab import files
import subprocess
import cv2
import math
from torch.utils.data import Dataset, DataLoader

class TranscriptDataset(Dataset):
    def __init__(self, segments):
        self.segments = [s for s in segments if s and len(s.split()) >= 10]

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        return self.segments[idx]

class LectureScreenshotsSummarizer:
    def __init__(self):
        device = 0 if torch.cuda.is_available() else -1
        if device == 0:
            print(f"Using GPU for summarization: {torch.cuda.get_device_name(0)}")
        else:
            print("Using CPU for summarization")
        self.summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6", device=device)

    def get_transcript(self, video_id):
        try:
            # Try English transcript first
            transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])
            return transcript
        except youtube_transcript_api.TranscriptsDisabled:
            return "Transcripts disabled for this video"
        except youtube_transcript_api.NoTranscriptFound:
            try:
                # Fallback to Hindi transcript and translate to English
                print("No English transcript found. Fetching Hindi transcript and translating to English...")
                transcript = YouTubeTranscriptApi.list_transcripts(video_id).find_transcript(['hi']).translate('en').fetch()
                return transcript
            except Exception as e:
                return f"Error fetching transcript: {str(e)}"

    def clean_text(self, text):
        text = re.sub(r'[^\w\s.,-]', '', text)
        return re.sub(r'\s+', ' ', text).strip()

    def download_video(self, youtube_url):
        try:
            output_template = os.path.join(os.getcwd(), "video.%(ext)s")
            cmd = [
                "yt-dlp",
                "-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
                "-o", output_template,
                youtube_url
            ]
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            video_path = os.path.join(os.getcwd(), "video.mp4")
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            duration = frame_count / fps
            cap.release()
            return video_path, duration
        except Exception as e:
            return f"Error downloading video: {str(e)}", None

    def extract_frames(self, video_path, interval=20):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_interval = int(fps * interval)
        frames = []
        timestamps = []
        count = 0

        print("Extracting frames...")
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if count % frame_interval == 0:
                frame_path = f"frame_{count // frame_interval}.jpg"
                cv2.imwrite(frame_path, frame)
                frames.append(frame_path)
                timestamps.append(count / fps)
            count += 1
        cap.release()
        return frames, timestamps

    def segment_transcript(self, transcript, timestamps):
        if isinstance(transcript, str):
            return [transcript] * len(timestamps)  # Fallback if no transcript

        segments = []
        for i in range(len(timestamps)):
            start_time = timestamps[i]
            end_time = timestamps[i + 1] if i + 1 < len(timestamps) else float('inf')
            if isinstance(transcript[0], dict):  # Regular transcript
                segment_text = " ".join(
                    [entry['text'] for entry in transcript if start_time <= entry['start'] < end_time]
                )
            else:  # Translated transcript (FetchedTranscriptSnippet)
                segment_text = " ".join(
                    [entry.text for entry in transcript if start_time <= entry.start < end_time]
                )
            segments.append(segment_text)
        return segments

    def summarize_segments(self, segments):
        dataset = TranscriptDataset(segments)
        dataloader = DataLoader(dataset, batch_size=4, shuffle=False)
        summaries = [""] * len(segments)

        if len(dataset) > 0:
            print(f"Summarizing {len(dataset)} segments in batches...")
            idx_offset = 0
            for batch_idx, batch in enumerate(dataloader):
                print(f"Processing batch {batch_idx + 1}/{len(dataloader)}...")
                batch_summaries = self.summarizer(
                    batch,
                    max_length=100,
                    min_length=10,
                    do_sample=False,
                    truncation=True
                )
                for i, summary in enumerate(batch_summaries):
                    summaries[idx_offset + i] = summary['summary_text']
                idx_offset += len(batch)

        # Fill in short or empty segments
        for i, segment in enumerate(segments):
            if not summaries[i]:
                summaries[i] = segment if segment and len(segment.split()) >= 10 else "No significant content in this segment"

        return summaries

    def generate_pdf(self, video_id, frames, summaries, output_filename):
        pdf_path = os.path.join(os.getcwd(), f"{output_filename}.pdf")
        doc = SimpleDocTemplate(pdf_path, pagesize=letter)
        styles = getSampleStyleSheet()

        story = []
        story.append(Paragraph("YouTube Lecture Timeline Summary", styles['Title']))
        story.append(Paragraph(f"Video ID: {video_id}", styles['Normal']))
        story.append(Spacer(1, 12))

        for i, (frame, summary) in enumerate(zip(frames, summaries)):
            story.append(Paragraph(f"Time: {i*20}s - {(i+1)*20}s", styles['Heading2']))
            story.append(Image(frame, width=200, height=150))
            cleaned_summary = self.clean_text(summary)
            story.append(Paragraph(cleaned_summary, styles['Normal']))
            story.append(Spacer(1, 12))

        print("Generating PDF with screenshots...")
        doc.build(story)
        print(f"PDF generated at {pdf_path}. Downloading now...")
        files.download(pdf_path)

    def process_video(self, youtube_url, output_filename="lecture_timeline"):
        video_id = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11}).*', youtube_url)
        if not video_id:
            return "Invalid YouTube URL"

        video_id = video_id.group(1)

        print("Fetching transcript...")
        transcript = self.get_transcript(video_id)
        if isinstance(transcript, str) and "Error" in transcript:
            print(f"Transcript issue: {transcript}")

        print("Downloading video...")
        video_result, duration = self.download_video(youtube_url)
        if isinstance(video_result, str) and "Error" in video_result:
            return video_result

        video_path = video_result
        frames, timestamps = self.extract_frames(video_path)
        print("Segmenting transcript...")
        segments = self.segment_transcript(transcript, timestamps)
        summaries = self.summarize_segments(segments)

        output_filename = output_filename.replace(" ", "_")
        self.generate_pdf(video_id, frames, summaries, output_filename)
        return f"PDF timeline generated and saved as {os.path.join(os.getcwd(), output_filename)}.pdf"

def main():
    summarizer = LectureScreenshotsSummarizer()
    youtube_url = input("Please enter the YouTube lecture URL: ")
    filename = input("Enter the desired PDF filename (without extension, default is 'lecture_timeline'): ") or "lecture_timeline"
    output_filename = filename.strip()
    result = summarizer.process_video(youtube_url, output_filename)
    print(result)

if __name__ == "__main__":
    main()

Using GPU for summarization: Tesla T4


Device set to use cuda:0


Please enter the YouTube lecture URL: https://youtu.be/5DSnWRHrv6U?si=QNAWmsHTrYBymgX0
Enter the desired PDF filename (without extension, default is 'lecture_timeline'): calender
Fetching transcript...
No English transcript found. Fetching Hindi transcript and translating to English...
Extracting frames...


Your max_length is set to 100, but your input_length is only 48. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=24)


Segmenting transcript...
Summarizing 246 segments in batches...
Processing batch 1/62...


...

Your max_length is set to 100, but your input_length is only 66. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)
Your max_length is set to 100, but your input_length is only 56. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=28)
Your max_length is set to 100, but your input_length is only 92. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=46)
Your max_length is set to 100, but your input_length is only 72. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=36)


Processing batch 62/62...
Generating PDF with screenshots...
PDF generated at /content/calender.pdf. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PDF timeline generated and saved as /content/calender.pdf
